# Класифікація спаму: Email Spam Detection

Ноутбук для запуску в Google Colab (**Runtime -> Run all**), GPU не потрібен.

Порівнюємо два способи перетворити текст у числа:
- **Підрахунок слів:** BoW (`CountVectorizer`) і TF-IDF (`TfidfVectorizer`);
- **Попередньо навчені ембединги:** GloVe (усереднений вектор слів у повідомленні).

Для кожного варіанту навчаємо модель і порівнюємо їх на валідаційному наборі. Висновки та аналіз - у кінці ноутбука.

## 1. Імпорт бібліотек

In [ ]:
!pip install -q kagglehub gensim

import random
import re
from pathlib import Path

import gensim.downloader as api
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Завантаження даних та попередній аналіз

Датасет завантажується через `kagglehub` (публічний, ключ Kaggle не потрібен). Колонки в оригіналі називаються `v1` (мітка) і `v2` (текст), перейменовуємо їх. Повні дублікати прибираємо, інакше те саме повідомлення може потрапити і в train, і в validation, і метрики будуть завищені.

In [ ]:
dataset_path = Path(kagglehub.dataset_download('shantanudhakadd/email-spam-detection-dataset-classification'))
csv_path = next(dataset_path.rglob('*.csv'))
print('File:', csv_path)

df = pd.read_csv(csv_path, encoding='latin-1').iloc[:, :2]
df.columns = ['label', 'text']
print('Rows before cleaning:', len(df))
df = df.dropna().drop_duplicates().reset_index(drop=True)
print('Rows after removing duplicates:', len(df))

df['target'] = (df['label'] == 'spam').astype(int)  # 1 = spam, 0 = ham
df.head()

In [ ]:
counts = df['label'].value_counts()
print(counts)
print((counts / len(df)).round(3))

df['length'] = df['text'].str.len()
print()
print(df.groupby('label')['length'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot.bar(ax=axes[0], color=['tab:blue', 'tab:red'], rot=0)
axes[0].set_title('Class distribution')
for label, color in (('ham', 'tab:blue'), ('spam', 'tab:red')):
    axes[1].hist(df.loc[df['label'] == label, 'length'], bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title('Message length (characters)')
axes[1].legend()
fig.tight_layout()
plt.show()

for label in ('ham', 'spam'):
    print(f'--- {label} examples ---')
    for text in df.loc[df['label'] == label, 'text'].sample(3, random_state=SEED):
        print('*', text)

**Що видно з аналізу:** класи **незбалансовані** (спаму приблизно 13%), тому одна лише accuracy може вводити в оману: модель, яка завжди відповідає `ham`, матиме ~87% точності. Спам у середньому довший за ham.

## 3. Попередня обробка тексту та розділення даних

Обробка: нижній регістр, видалення всього крім латинських літер, видалення стоп-слів і односимвольних слів. Ця обробка не залежить від даних, тому її можна застосувати до всього датасету до розділення.

Розділення: 80% train / 20% validation, стратифіковано, щоб частка спаму в обох наборах була однаковою. Усі векторизатори далі навчаються **тільки на train**, щоб не було витоку інформації з validation.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    return ' '.join(w for w in text.split() if w not in ENGLISH_STOP_WORDS and len(w) > 1)


df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].head(3).to_string())

X_train, X_val, y_train, y_val = train_test_split(
    df['clean_text'], df['target'], test_size=0.2, stratify=df['target'], random_state=SEED,
)
print()
print(f'train: {len(X_train)} (spam share {y_train.mean():.3f})')
print(f'val:   {len(X_val)} (spam share {y_val.mean():.3f})')

## 4. BoW та TF-IDF

- `max_features=5000`: лишаємо 5000 найчастіших ознак, щоб обмежити розмір словника і шум від рідкісних слів;
- `ngram_range=(1, 2)`: окрім окремих слів беремо пари слів (наприклад, `free entry`, `call now`);
- `min_df=2`: ігноруємо ознаки, які зустрілись лише в одному документі.

In [ ]:
bow = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)

X_train_bow = bow.fit_transform(X_train)
X_val_bow = bow.transform(X_val)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print('BoW:   ', X_train_bow.shape, X_val_bow.shape)
print('TF-IDF:', X_train_tfidf.shape, X_val_tfidf.shape)

## 5. Попередньо навчені ембединги (GloVe)

Беремо `glove-wiki-gigaword-100` (100-вимірні вектори, навчені на Wikipedia і Gigaword). Завантаження займає близько хвилини (~130 МБ). Вектор повідомлення - це **середнє** векторів його слів; слова, яких немає у словнику GloVe, пропускаються, а якщо не лишилось жодного слова - вектор нульовий.

In [ ]:
glove = api.load('glove-wiki-gigaword-100')


def text_to_vector(text, model=glove):
    vectors = [model[w] for w in text.split() if w in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)


X_train_emb = np.vstack([text_to_vector(t) for t in X_train])
X_val_emb = np.vstack([text_to_vector(t) for t in X_val])
print('Embeddings:', X_train_emb.shape, X_val_emb.shape)

oov_share = np.mean([all(w not in glove for w in t.split()) for t in X_train if t])
print(f'Messages with no known words: {oov_share:.2%}')

## 6. Побудова та навчання моделей

Для всіх трьох представлень використовуємо `LogisticRegression`, щоб порівняння було чесним: відрізняється лише спосіб перетворення тексту. Для ембедингів додаємо `StandardScaler`, бо логістична регресія краще збігається на нормованих ознаках.

In [ ]:
experiments = {
    'BoW + LogReg': (LogisticRegression(max_iter=1000, random_state=SEED), X_train_bow, X_val_bow),
    'TF-IDF + LogReg': (LogisticRegression(max_iter=1000, random_state=SEED), X_train_tfidf, X_val_tfidf),
    'GloVe + LogReg': (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
        X_train_emb,
        X_val_emb,
    ),
}

for name, (model, X_tr, _) in experiments.items():
    model.fit(X_tr, y_train)
    print('Trained:', name)

## 7. Оцінка моделей

**Вибір метрик.** Через дисбаланс класів дивимось не лише на accuracy:
- **AUC** не залежить від порогу класифікації і показує, наскільки добре модель ранжує спам вище за ham;
- **Accuracy** - загальна частка правильних відповідей;
- **Precision / Recall / F1 для класу spam**: precision показує, скільки легітимних листів ми помилково відправили у спам (це дуже шкідливо), recall - скільки спаму пропустили.

In [ ]:
def evaluate(name, model, X_val_features):
    pred = model.predict(X_val_features)
    if hasattr(model, 'predict_proba'):
        score = model.predict_proba(X_val_features)[:, 1]
    else:
        score = model.decision_function(X_val_features)
    return {
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'auc': roc_auc_score(y_val, score),
        'precision_spam': precision_score(y_val, pred),
        'recall_spam': recall_score(y_val, pred),
        'f1_spam': f1_score(y_val, pred),
    }, pred, score


rows, preds, scores = [], {}, {}
for name, (model, _, X_v) in experiments.items():
    row, preds[name], scores[name] = evaluate(name, model, X_v)
    rows.append(row)

metrics_table = pd.DataFrame(rows).set_index('model')
metrics_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
for name in experiments:
    fpr, tpr, _ = roc_curve(y_val, scores[name])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={metrics_table.loc[name, "auc"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set(title='ROC curves (validation)', xlabel='False positive rate', ylabel='True positive rate')
axes[0].legend(loc='lower right')

for ax, name in zip(axes[1:], experiments):
    m = confusion_matrix(y_val, preds[name])
    ax.imshow(m, cmap='Blues')
    ax.set(
        title=name,
        xlabel='Predicted',
        ylabel='True',
        xticks=[0, 1],
        yticks=[0, 1],
        xticklabels=['ham', 'spam'],
        yticklabels=['ham', 'spam'],
    )
    for r in range(2):
        for c in range(2):
            ax.text(c, r, m[r, c], ha='center', va='center', fontsize=14)
fig.tight_layout()
plt.show()

Найінформативніші слова для TF-IDF моделі (найбільші додатні коефіцієнти - ознаки спаму, найбільші від'ємні - ознаки ham):

In [ ]:
coefs = experiments['TF-IDF + LogReg'][0].coef_[0]
features = np.array(tfidf.get_feature_names_out())
order = np.argsort(coefs)
print('Spam indicators:', ', '.join(features[order[-15:][::-1]]))
print('Ham indicators: ', ', '.join(features[order[:15]]))

## 8. Вдосконалення моделей (опціонально)

Пробуємо інші алгоритми (`MultinomialNB`, `LinearSVC`, `RandomForestClassifier`) і підбираємо гіперпараметри TF-IDF + LogReg через `GridSearchCV` (5-fold крос-валідація **тільки на train**, метрика AUC). Validation використовується лише для фінальної оцінки.

In [ ]:
other_models = {
    'BoW + NaiveBayes': (MultinomialNB(), X_train_bow, X_val_bow),
    'TF-IDF + LinearSVC': (LinearSVC(random_state=SEED), X_train_tfidf, X_val_tfidf),
    'TF-IDF + RandomForest': (
        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED), X_train_tfidf, X_val_tfidf,
    ),
    'GloVe + RandomForest': (
        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED), X_train_emb, X_val_emb,
    ),
}

extra_rows = []
for name, (model, X_tr, X_v) in other_models.items():
    model.fit(X_tr, y_train)
    extra_rows.append(evaluate(name, model, X_v)[0])

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
])
param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1, 10, 100],
}
grid = GridSearchCV(pipeline, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print(f'Best CV AUC: {grid.best_score_:.4f}')
extra_rows.append(evaluate('TF-IDF + LogReg (tuned)', grid.best_estimator_, X_val)[0])

final_table = pd.concat([metrics_table, pd.DataFrame(extra_rows).set_index('model')])
final_table.sort_values('auc', ascending=False).round(4)

## 9. Висновки

Результати на валідаційному наборі (1034 повідомлення, з них ~131 спаму). При повторному запуску цифри можуть трохи відрізнятися.

| Модель | Accuracy | AUC | Precision (spam) | Recall (spam) | F1 (spam) |
|---|---|---|---|---|---|
| BoW + LogReg | 0.9749 | 0.9898 | 0.9817 | 0.8168 | 0.8917 |
| TF-IDF + LogReg | 0.9555 | 0.9939 | 0.9885 | 0.6565 | 0.7890 |
| GloVe + LogReg | 0.9304 | 0.9534 | 0.7565 | 0.6641 | 0.7073 |
| BoW + NaiveBayes | 0.9894 | 0.9806 | 0.9839 | 0.9313 | 0.9569 |
| TF-IDF + LinearSVC | 0.9816 | 0.9949 | 0.9828 | 0.8702 | 0.9231 |
| TF-IDF + RandomForest | 0.9758 | 0.9867 | 0.9907 | 0.8168 | 0.8954 |
| TF-IDF + LogReg (tuned) | 0.9768 | 0.9942 | 0.9735 | 0.8397 | 0.9016 |
| GloVe + RandomForest | 0.9516 | 0.9635 | 0.9765 | 0.6336 | 0.7685 |

Підбір гіперпараметрів: `C=10`, `max_features=5000`, `ngram_range=(1, 2)`, CV AUC на train = 0.9891 (на validation вийшло 0.9942, тобто великої різниці немає і перенавчання теж немає).

### 1. Яка модель працює краще

- Усі моделі на **BoW і TF-IDF** мають accuracy 0.95-0.99 і AUC 0.98-0.995, це набагато вище за орієнтовні 82-85% і 0.84-0.87 з умови завдання. Це не ознака перенавчання: validation відокремлений від train, дублікати прибрані, а спам у цьому датасеті дуже характерний (`free`, `txt`, `call`, `prize`, `claim`, номери телефонів), тому задача відносно проста.
- **Найкращий за AUC:** TF-IDF + LinearSVC (0.9949), далі налаштована TF-IDF + LogReg (0.9942) і TF-IDF + LogReg зі стандартними параметрами (0.9939).
- **Найкращий за accuracy і F1:** BoW + NaiveBayes (0.9894 і 0.9569). Він знаходить 93% спаму (recall 0.93) при precision 0.98, хоча його AUC (0.9806) нижча, ніж у TF-IDF моделей.
- **Ембединги GloVe поступаються:** GloVe + LogReg має accuracy 0.9304 і AUC 0.9534, а головне - precision лише 0.76 (приблизно 28 легітимних листів помилково потрапили у спам). GloVe + RandomForest має високий precision (0.98), але знаходить лише 63% спаму.
- **Чому TF-IDF + LogReg має високий AUC, але низький recall (0.66).** AUC не залежить від порогу: модель добре ранжує спам вище за ham. Але зі стандартним `C=1` ймовірності для TF-IDF стискаються і при порозі 0.5 частина спаму лишається нижче порогу. Після підбору `C=10` recall зріс з 0.66 до 0.84, а F1 з 0.79 до 0.90, хоча AUC майже не змінилась. Тому accuracy і AUC самі по собі не показують проблему, потрібно дивитись ще на recall і F1.
- Різниця BoW і TF-IDF за AUC невелика (0.9898 проти 0.9939), TF-IDF трохи краще, бо знижує вагу частих слів.
- Через дисбаланс класів (~13% спаму) порівнювати моделі краще за **AUC і F1 для spam**, ніж за accuracy: наприклад, модель, що завжди відповідає ham, мала б accuracy ~87%.

### 2. Переваги та недоліки підходів

**BoW / TF-IDF**
- (+) Прості, швидкі, добре інтерпретуються (видно, які слова вказують на спам).
- (+) Чудово працюють, коли клас визначається ключовими словами, як у спамі.
- (-) Не розуміють значення слів: `win` і `won` для них різні ознаки, а синоніми не пов'язані.
- (-) Не враховують порядок слів (частково виправляють n-грами), вектори розріджені та великі.
- (-) Невідомі слова на нових даних ігноруються.

**Попередньо навчені ембединги (GloVe)**
- (+) Передають семантичну близькість слів, вектор компактний (100 вимірів).
- (+) Працюють зі словами, яких не було в train, якщо вони є в словнику GloVe.
- (-) Усереднення втрачає порядок слів і розмиває важливі слова (одне слово `free` губиться серед багатьох нейтральних).
- (-) GloVe навчений на новинах і Wikipedia, а не на SMS: сленг, скорочення, помилки та числа (`txt`, `u`, `2day`) не покриті або потрапляють у вектор неправильно.
- (-) Вектори важко інтерпретувати, потрібно завантажувати велику модель.

### 3. Що ще можна зробити

- Не видаляти цифри і спеціальні символи повністю: наявність номера телефону, `£`, `$` або великої кількості `!` сильно вказує на спам. Можна додати такі ознаки окремо (довжина, кількість цифр).
- Використати `class_weight='balanced'` або змінити поріг класифікації, щоб збільшити recall спаму (особливо для TF-IDF + LogReg і GloVe).
- Підібрати гіперпараметри для інших моделей (`LinearSVC`, `NaiveBayes`), а не лише для логістичної регресії.
- Брати ембединги, навчені на ближчих даних (наприклад, `glove-twitter-100`), або використати TF-IDF-зважене усереднення замість простого середнього.
- Спробувати fine-tuning трансформера (BERT/DistilBERT): він враховує контекст і порядок слів.

### Загальний підсумок

Для детекції спаму прості моделі на BoW і TF-IDF виявились кращими за варіант з усередненими GloVe-ембедингами: найвищий AUC у TF-IDF + LinearSVC (0.9949), найвищі accuracy і F1 у BoW + NaiveBayes (0.9894 і 0.9569), а найкращий GloVe (LogReg) має лише AUC 0.9534 і precision 0.76. Спам визначається ключовими словами, які розріджені вектори передають напряму, а усереднені GloVe-вектори їх розмивають. Також видно, що оцінювати модель тільки за accuracy або AUC недостатньо: потрібні ще precision, recall і F1 для класу spam.
